In [ ]:
"""
005-diet_logging_paper.py
=========================
严格复现论文数据处理逻辑（见论文 Methods 原文）：

  "Diet was measured as the daily mean consumption of 322 foods,
   computed from a maximal period of 16 days of real-time self-logging.
   Logging days with less than 500 kcal were excluded from the analysis.
   Foods that were not logged by at least 5% of all participants were not included.
   Logging data were clipped at 10 standard deviations, as computed using
   the middle 99th percentile of the data."

与 fixed notebook 的主要差异：
  ① max_days = 16，且按日期排序取【前 16 天】（原文逻辑，非随机抽样）
  ② 5% 覆盖率分母 = 过滤前的全体参与者总数（稳定基准，FIX3 保留）
  ③ Clipping = middle 99th percentile 计算 mean±10SD（FIX 保留）
  ④ 日均值 = sum / n_valid_days（没吃算 0，FIX1 保留）
  ⑤ NaN 填 0 后输出宽表（FIX4 保留）
  ⑥ raw 路径与 events 路径均独立保留，不互相覆盖（FIX2 保留）
  ⑦ 路径B修复重名 collection_date 列 bug

输出：
  pheno_data/202512/005-diet_logging_imputed.parquet      food_id 宽表（论文复现）
  pheno_data/202512/005-diet_mean_long.parquet             long format（中间结果）
  pheno_data/202512/005-diet_nutrients.parquet             营养素聚合表（衰老时钟输入）
"""

import sys
import pandas as pd
import numpy as np

sys.path.append("/home/ec2-user/studies/HUIFA-RNA-SEQ-new/phenoAI/")
from phenokit.preprocessing.qc import MissingValueAnalyzer   # noqa: E402
from phenokit.preprocessing.io import read_parquet            # noqa: E402


# =============================================================================
# 0. 数据加载
# =============================================================================
pfile = "/home/ec2-user/studies/hpp_datasets/diet_logging/diet_logging.parquet"
df_diet_logging = read_parquet(pfile)
print(f"diet_logging          : {df_diet_logging.shape}")

pfile = "/home/ec2-user/studies/hpp_datasets/diet_logging/diet_logging_events.parquet"
df_dlog_events = read_parquet(pfile)
print(f"diet_logging_events   : {df_dlog_events.shape}")

pfile = "/home/ec2-user/studies/hpp_datasets/diet_logging/raw_diet_logging_events.parquet"
df_raw_dlog_events = read_parquet(pfile)
df_raw_dlog_events.drop(columns=["research_stage"], inplace=True)
print(f"raw_diet_logging_events: {df_raw_dlog_events.shape}")


# =============================================================================
# 1. 核心处理函数
# =============================================================================

def clip_by_10sd_middle99(s: pd.Series) -> pd.Series:
    """
    论文原文：'clipped at 10 standard deviations, as computed using
               the middle 99th percentile of the data'
    实现：用 0.5%–99.5% 分位数区间内的数据计算 mean 和 SD，
          然后将全部数据截断到 [mean - 10*SD, mean + 10*SD]。
    """
    s = s.astype(float)
    s_nonan = s.dropna()
    if len(s_nonan) < 50:          # 样本太少，跳过 clipping
        return s

    lo = s_nonan.quantile(0.005)
    hi = s_nonan.quantile(0.995)
    trimmed = s_nonan[(s_nonan >= lo) & (s_nonan <= hi)]

    if trimmed.std(ddof=0) == 0 or len(trimmed) < 10:
        return s

    mu = trimmed.mean()
    sd = trimmed.std(ddof=0)
    return s.clip(mu - 10 * sd, mu + 10 * sd)


def process_diet_logs(
    df: pd.DataFrame,
    max_days_per_participant: int = 16,      # 论文：maximal period of 16 days
    min_kcal_per_day: float = 500.0,         # 论文：<500 kcal days excluded
    min_participant_frac: float = 0.05,      # 论文：at least 5% of all participants
    total_participants: int = None,          # FIX3：稳定分母，过滤前传入
    value_cols=None,
):
    """
    严格复现论文的饮食数据处理流程。

    参数
    ----
    df                      : 原始饮食记录（long format）
    max_days_per_participant: 每人每 stage 最多使用的天数（论文 = 16，按日期排序取前N天）
    min_kcal_per_day        : 每天最低有效热量（论文 = 500 kcal）
    min_participant_frac    : 食物最低覆盖参与者比例（论文 = 5%）
    total_participants      : 覆盖率过滤的分母；None 则用过滤后数据计算（不推荐）
    value_cols              : 需要聚合的数值列；None 则自动选 8 个营养素列

    返回
    ----
    participant_food_mean : DataFrame  (participant, stage, food) × 日均营养素
    meta                  : DataFrame  每个 participant-stage 的有效天数/食物数统计
    daily_food            : DataFrame  每天每食物的汇总值（已 clipping，中间结果）
    """
    df = df.copy()

    # ── 步骤 1：解析日期 ──────────────────────────────────────────────────────
    if "collection_date" in df.columns:
        df["date"] = pd.to_datetime(df["collection_date"]).dt.date
    elif "local_timestamp" in df.columns:
        df["date"] = pd.to_datetime(df["local_timestamp"]).dt.date
    elif "collection_timestamp" in df.columns:
        df["date"] = pd.to_datetime(df["collection_timestamp"]).dt.date
    else:
        raise ValueError(
            "找不到日期字段：collection_date / local_timestamp / collection_timestamp"
        )

    # ── 步骤 2：确定数值列 ────────────────────────────────────────────────────
    if value_cols is None:
        candidate_cols = [
            "weight_g", "calories_kcal",
            "carbohydrate_g", "lipid_g", "protein_g",
            "sodium_mg", "alcohol_g", "dietary_fiber_g",
        ]
        value_cols = [c for c in candidate_cols if c in df.columns]

    required = {"participant_id", "food_id", "date", "research_stage"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"缺少必要列: {missing}")
    if "calories_kcal" not in df.columns:
        raise ValueError("需要 calories_kcal 才能做 <500 kcal/day 过滤")

    # ── 步骤 3：剔除每日总热量 <500 kcal 的天 ────────────────────────────────
    # 论文：'Logging days with less than 500 kcal were excluded'
    day_kcal = (
        df.groupby(["participant_id", "research_stage", "date"], as_index=False)
          ["calories_kcal"].sum()
          .rename(columns={"calories_kcal": "day_kcal"})
    )
    valid_days = day_kcal[day_kcal["day_kcal"] >= min_kcal_per_day].copy()

    # ── 步骤 4：每人每 stage 最多保留前 16 天（按日期升序） ──────────────────
    # 论文：'maximal period of 16 days'
    # 取【最早的 N 天】复现原文，保持与论文一致
    if max_days_per_participant is not None:
        valid_days = (
            valid_days
            .sort_values("date")                            # 按日期升序
            .groupby(["participant_id", "research_stage"], group_keys=False)
            .head(max_days_per_participant)                 # 取前 N 天
        )

    # 将有效天回连到明细日志
    df = df.merge(
        valid_days[["participant_id", "research_stage", "date"]],
        on=["participant_id", "research_stage", "date"],
        how="inner",
    )

    # ── 步骤 5：食物覆盖率过滤（>=5% 参与者） ────────────────────────────────
    # 论文：'Foods that were not logged by at least 5% of all participants
    #        were not included'
    # FIX3：使用传入的 total_participants 作为稳定分母，避免过滤后分母收缩
    n_participants = (
        total_participants
        if total_participants is not None
        else df["participant_id"].nunique()
    )
    food_coverage = df.groupby("food_id")["participant_id"].nunique()
    keep_foods    = food_coverage[
        food_coverage >= n_participants * min_participant_frac
    ].index
    df = df[df["food_id"].isin(keep_foods)].copy()

    # ── 步骤 6：同一天同一食物多次记录 → 求和 ────────────────────────────────
    daily_food = (
        df.groupby(
            ["participant_id", "research_stage", "date", "food_id"],
            as_index=False,
        )[value_cols].sum()
    )

    # ── 步骤 7：Clipping（论文：10 SD，基于 middle 99th percentile） ─────────
    for col in value_cols:
        daily_food[col] = (
            daily_food.groupby("food_id")[col]
                      .transform(clip_by_10sd_middle99)
        )

    # ── 步骤 8：计算日均值（没吃该食物的天计为 0） ───────────────────────────
    # 论文：'daily mean consumption of 322 foods'
    # = 该食物在所有有效天的总摄入量 / 总有效天数（未记录的天算 0）
    n_valid_days = (
        daily_food.groupby(["participant_id", "research_stage"])["date"]
                  .nunique()
                  .rename("n_valid_days")
                  .reset_index()
    )

    pf_sum = (
        daily_food
        .groupby(["participant_id", "research_stage", "food_id"], as_index=False)
        [value_cols].sum()
    )
    pf_sum = pf_sum.merge(
        n_valid_days, on=["participant_id", "research_stage"], how="left"
    )
    for col in value_cols:
        pf_sum[col] = pf_sum[col] / pf_sum["n_valid_days"]   # 没吃的天算 0
    participant_food_mean = pf_sum.drop(columns=["n_valid_days"])

    # ── meta 统计 ─────────────────────────────────────────────────────────────
    meta = (
        pd.concat(
            [
                daily_food.groupby(["participant_id", "research_stage"])["date"]
                          .nunique().rename("n_days"),
                daily_food.groupby(["participant_id", "research_stage"])["food_id"]
                          .nunique().rename("n_foods"),
            ],
            axis=1,
        ).reset_index()
    )

    return participant_food_mean, meta, daily_food


# =============================================================================
# 2. 路径 A：df_dlog_events
# =============================================================================

# FIX3：在任何过滤之前记录总参与者数，作为 5% 覆盖率的稳定分母
total_n_participants_A = df_dlog_events["participant_id"].nunique()
print(f"\n[路径A] 总参与者数（覆盖率分母）: {total_n_participants_A}")

df_diet_mean_A, meta_A, df_daily_A = process_diet_logs(
    df_dlog_events,
    max_days_per_participant=16,              # 论文：前 16 天
    total_participants=total_n_participants_A,
)
print(f"[路径A] process_diet_logs 输出: {df_diet_mean_A.shape}")
print(f"[路径A] 有效天数分布:\n{meta_A['n_days'].describe()}")
print(f"[路径A] 食物种数分布:\n{meta_A['n_foods'].describe()}")

# 补充 short_food_name
df_id_name_A = df_dlog_events[["food_id", "short_food_name"]].drop_duplicates()
df_diet_mean_A = pd.merge(df_diet_mean_A, df_id_name_A, on="food_id", how="left")

# 过滤掉 short_food_name 为空的行（无法识别的食物）
df_diet_filtered_A = df_diet_mean_A.dropna(subset=["short_food_name"]).copy()
print(f"[路径A] 过滤后: {df_diet_filtered_A.shape[0]} 行，"
      f"食物种数: {df_diet_filtered_A['food_id'].nunique()}")


# =============================================================================
# 3. 路径 B：df_raw_dlog_events（附 food_items 白名单过滤）
# =============================================================================

# 将 raw 数据与 research_stage 关联
keys = ["participant_id", "collection_day"]

right = df_diet_logging[["participant_id", "collection_date", "research_stage"]].copy()
right["collection_day"] = pd.to_datetime(right["collection_date"], utc=True).dt.date
right = right.drop(columns=["collection_date"]).drop_duplicates(subset=keys)

left = df_raw_dlog_events.copy()
left["collection_day"] = pd.to_datetime(left["collection_date"], utc=True).dt.date
left = left.drop(columns=["collection_date"])   # 删除原列，避免 merge 后出现两个同名列

df_raw_merged = left.merge(right, on=keys, how="left", validate="m:1")
df_raw_merged.dropna(subset=["research_stage"], inplace=True)
df_raw_merged.rename(columns={"collection_day": "collection_date"}, inplace=True)
print(f"\n[路径B] raw 数据合并 research_stage 后: {df_raw_merged.shape[0]} 行")

total_n_participants_B = df_raw_merged["participant_id"].nunique()
print(f"[路径B] 总参与者数（覆盖率分母）: {total_n_participants_B}")

df_diet_mean_B, meta_B, df_daily_B = process_diet_logs(
    df_raw_merged,
    max_days_per_participant=16,
    total_participants=total_n_participants_B,
)
print(f"[路径B] process_diet_logs 输出: {df_diet_mean_B.shape}")

# 补充 short_food_name
df_id_name_B = df_raw_dlog_events[["food_id", "short_food_name"]].drop_duplicates()
df_diet_mean_B = pd.merge(df_diet_mean_B, df_id_name_B, on="food_id", how="left")

# food_items 白名单过滤（路径B特有）
df_food_items = pd.read_csv(
    "data/005-diet_logging_items.csv",
    skiprows=1, header=None, names=["food_name", "n_males", "n_females"],
)
df_diet_filtered_B = df_diet_mean_B[
    df_diet_mean_B["short_food_name"].isin(df_food_items["food_name"].values)
].dropna(subset=["short_food_name"]).copy()
print(f"[路径B] 白名单过滤后: {df_diet_filtered_B.shape[0]} 行，"
      f"食物种数: {df_diet_filtered_B['food_id'].nunique()}")


# =============================================================================
# 4. 选择路径，构建宽表
# =============================================================================

# ── 在此选择路径 ─────────────────────────────────────────────────────────────
# 路径A（df_dlog_events，标准化数据，推荐）：
df_diet_selected = df_diet_filtered_A.copy()
df_daily_selected = df_daily_A

# 路径B（raw + food_items 白名单，如需切换取消注释下两行）：
# df_diet_selected  = df_diet_filtered_B.copy()
# df_daily_selected = df_daily_B
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n[宽表] 使用数据: {df_diet_selected.shape[0]} 行")

NUTR_COLS = [
    "weight_g", "calories_kcal", "carbohydrate_g", "lipid_g", "protein_g",
    "sodium_mg", "alcohol_g", "dietary_fiber_g",
]
nutr_cols = [c for c in NUTR_COLS if c in df_diet_selected.columns]

# 每个 participant-stage 的最早记录日期（用于与其他数据集对齐）
first_dt = (
    df_daily_selected
    .groupby(["participant_id", "research_stage"])["date"]
    .min()
    .rename("collection_date")
    .reset_index()
)

# pivot → 宽表：列 = nutritional_value__food_id
wide = df_diet_selected.pivot(
    index=["participant_id", "research_stage"],
    columns="food_id",
    values=nutr_cols,
)
wide.columns = [f"{nutr}__{food}" for nutr, food in wide.columns]
wide = wide.reset_index()

# NaN 填 0（论文中"没吃该食物的天"已在均值计算时算作 0，此处仅处理 pivot 产生的 NaN）
food_cols = [c for c in wide.columns if c not in ["participant_id", "research_stage"]]

sparsity = wide[food_cols].isna().mean()
print(f"[宽表] 稀疏度：中位数 {sparsity.median():.1%}，最大 {sparsity.max():.1%}，"
      f">90% 零值列 {(sparsity > 0.9).sum()} / {len(food_cols)}")

wide[food_cols] = wide[food_cols].fillna(0)

# 合并最早记录日期
df_diet_final = wide.merge(
    first_dt,
    on=["participant_id", "research_stage"],
    how="left",
    validate="1:1",
)

print(f"[宽表] 最终维度: {df_diet_final.shape}")


# =============================================================================
# 5. 数据质量核查
# =============================================================================

print("\n── 数据质量核查 ──────────────────────────────────────────────")
print(f"最终参与者数  : {df_diet_final['participant_id'].nunique()}")
print(f"research_stage: {df_diet_final['research_stage'].nunique()} 个")
print(f"\n有效天数分布（论文期望中位数接近 16）:")
print(meta_A["n_days"].describe().to_string())
print(f"\n各 research_stage 样本量:")
print(df_diet_final["research_stage"].value_counts().to_string())
print(f"\n食物种数（论文报告 322）: {df_diet_selected['food_id'].nunique()}")


# =============================================================================
# 6. 营养素聚合（衰老时钟核心输入）
# =============================================================================
# 在 food_id 层面的日均值（df_diet_selected）基础上，
# 对每个 participant-stage 把所有食物的营养素直接求和，
# 得到"每日总摄入量"。这是比 food_id 宽表更稳健的衰老时钟特征。

print("\n── 营养素聚合 ─────────────────────────────────────────────────")

KEY_COLS = ["participant_id", "research_stage"]
eps = 1e-6

# ── 6.1 每日总营养素（对所有食物求和） ───────────────────────────────────────
nutr_agg = (
    df_diet_selected
    .groupby(KEY_COLS, as_index=False)[nutr_cols]
    .sum()
)

# ── 6.2 衍生特征：供能比 & 纤维/碳水比 ──────────────────────────────────────
kcal = nutr_agg.get("calories_kcal", pd.Series(np.nan, index=nutr_agg.index))

if "protein_g" in nutr_agg.columns:
    nutr_agg["protein_energy_pct"]  = nutr_agg["protein_g"]      * 4 / (kcal + eps)
if "carbohydrate_g" in nutr_agg.columns:
    nutr_agg["carb_energy_pct"]     = nutr_agg["carbohydrate_g"] * 4 / (kcal + eps)
if "lipid_g" in nutr_agg.columns:
    nutr_agg["fat_energy_pct"]      = nutr_agg["lipid_g"]        * 9 / (kcal + eps)
if "alcohol_g" in nutr_agg.columns:
    nutr_agg["alcohol_energy_pct"]  = nutr_agg["alcohol_g"]      * 7 / (kcal + eps)
if {"dietary_fiber_g", "carbohydrate_g"}.issubset(nutr_agg.columns):
    nutr_agg["fiber_carb_ratio"]    = nutr_agg["dietary_fiber_g"] / (nutr_agg["carbohydrate_g"] + eps)

# ── 6.3 合并最早记录日期（与其他数据集对齐用） ──────────────────────────────
nutr_agg = nutr_agg.merge(first_dt, on=KEY_COLS, how="left")

# ── 6.4 合并有效天数（质量指标，建模时可作协变量） ───────────────────────────
n_days_meta = meta_A[KEY_COLS + ["n_days", "n_foods"]]
nutr_agg = nutr_agg.merge(n_days_meta, on=KEY_COLS, how="left")

# ── 6.5 质量检查 ──────────────────────────────────────────────────────────────
nutr_feat_cols = [c for c in nutr_agg.columns if c not in KEY_COLS + ["collection_date"]]
missing_pct    = nutr_agg[nutr_feat_cols].isna().mean()
print(f"营养素聚合表维度 : {nutr_agg.shape}")
print(f"特征列数         : {len(nutr_feat_cols)}")
print(f"缺失率 > 5% 的列 : {missing_pct[missing_pct > 0.05].to_dict()}")
print(f"\n各营养素描述统计:")
print(nutr_agg[nutr_feat_cols].describe().round(2).to_string())


# =============================================================================
# 7. 保存
# =============================================================================

# 7.1 food_id 宽表（论文复现，保留以供对比）
OUT_WIDE = "pheno_data/202512/005-diet_logging_imputed.parquet"
df_diet_final.reset_index(drop=True).to_parquet(OUT_WIDE)
print(f"\n已保存 food_id 宽表 : {OUT_WIDE}  {df_diet_final.shape}")

# 7.2 Long format（中间结果，供进一步特征工程）
OUT_LONG = "pheno_data/202512/005-diet_mean_long.parquet"
df_diet_selected.reset_index(drop=True).to_parquet(OUT_LONG)
print(f"已保存 long format  : {OUT_LONG}  {df_diet_selected.shape}")

# 7.3 营养素聚合表（衰老时钟直接输入）
OUT_NUTR = "pheno_data/202512/005-diet_nutrients.parquet"
nutr_agg.reset_index(drop=True).to_parquet(OUT_NUTR)
print(f"已保存营养素聚合表  : {OUT_NUTR}  {nutr_agg.shape}")

# 验证
df_check = pd.read_parquet(OUT_NUTR)
print(f"\n验证读取营养素表: {df_check.shape}")
print(df_check.head(3).to_string())